# PDE-Based Image Inpainting: Model Comparison

**Authors:** Blake Taylor, Fang Fang, Inzaghi Moniaga

This notebook demonstrates four PDE-based inpainting methods:

| Model | Conductivity G | Key Property |
|-------|----------------|---------------|
| **Harmonic** | G = 1 | Isotropic diffusion, blurs edges |
| **TV** | G = 1/\|∇u\| | Edge-preserving |
| **CDD** | G = g(\|κ\|)/\|∇u\| | Curvature-driven, connectivity principle |
| **QCDD** | G = g(\|κ\|) | Faster curvature-driven |

All models solve the steady-state PDE: **∇·(G∇u) = 0** in Ω, with **u = f** on ∂Ω

---

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter, binary_dilation

print("Libraries loaded successfully!")

## 1. Model Implementations

### 1.1 Harmonic Inpainting (Baseline)

Solves the Laplace equation **Δu = 0** via Gauss-Seidel iteration.

**Key fix:** Uses boundary-aware initialization (mean of mask boundary pixels) instead of global mean.

In [ ]:
def harmonic_inpaint(
    image: np.ndarray,
    mask: np.ndarray,
    max_iter: int = 5000,
    tol: float = 1e-5,
    verbose: bool = True,
) -> tuple:
    """
    Harmonic inpainting via Gauss-Seidel iteration.
    Uses boundary-aware initialization for better structure preservation.
    """
    u = image.copy().astype(np.float64)

    # FIXED: Initialize with boundary values, not global mean
    # This preserves structure information at the mask edges
    dilated = binary_dilation(mask)
    boundary = dilated & ~mask  # Non-mask pixels adjacent to mask

    if np.any(boundary):
        u[mask] = image[boundary].mean()
    elif np.any(~mask):
        u[mask] = image[~mask].mean()
    else:
        u[mask] = 0.5

    residuals = []

    for iteration in range(max_iter):
        u_old = u.copy()

        # Pad with edge replication
        u_pad = np.pad(u, 1, mode='edge')

        # 4-neighbor average (Laplacian = 0 steady state)
        u_new = 0.25 * (
            u_pad[1:-1, 2:]   +  # East
            u_pad[1:-1, :-2]  +  # West
            u_pad[:-2, 1:-1]  +  # North
            u_pad[2:, 1:-1]      # South
        )

        # Update only inside mask
        u[mask] = u_new[mask]
        u[~mask] = image[~mask]

        # Convergence check
        if np.any(mask):
            max_change = float(np.max(np.abs(u[mask] - u_old[mask])))
        else:
            max_change = 0.0
        residuals.append(max_change)

        if verbose and (iteration + 1) % 1000 == 0:
            print(f"  Harmonic Iter {iteration + 1:5d} | max Δu = {max_change:.6f}")

        if max_change < tol:
            if verbose:
                print(f"  Harmonic converged at iteration {iteration + 1}")
            break
    else:
        if verbose:
            print(f"  Harmonic reached max_iter={max_iter}")

    return u, residuals

print("✓ Harmonic inpainting defined")

### 1.2 Total Variation (TV) Inpainting

Uses conductivity **G = 1/|∇u|** for edge preservation.

In [ ]:
def tv_inpaint(
    image: np.ndarray,
    mask: np.ndarray,
    max_iter: int = 3000,
    eps: float = 0.01,
    tol: float = 1e-5,
    verbose: bool = True,
) -> tuple:
    """
    TV inpainting via weighted Gauss-Seidel on the steady-state equation.
    Uses boundary-aware initialization.
    """
    u = image.copy().astype(np.float64)

    # FIXED: Initialize with boundary values, not global mean
    dilated = binary_dilation(mask)
    boundary = dilated & ~mask

    if np.any(boundary):
        u[mask] = image[boundary].mean()
    elif np.any(~mask):
        u[mask] = image[~mask].mean()
    else:
        u[mask] = 0.5

    residuals = []
    eps_sq = eps ** 2

    for iteration in range(max_iter):
        u_old = u.copy()

        # Pad with edge replication
        u_pad = np.pad(u, 1, mode='edge')

        # Conductivity weights at the 4 half-points
        diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
        diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
        diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
        diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]

        G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
        G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
        G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
        G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)

        # Weighted average update
        u_E = u_pad[1:-1, 2:]
        u_W = u_pad[1:-1, :-2]
        u_N = u_pad[:-2, 1:-1]
        u_S = u_pad[2:, 1:-1]

        numerator = G_E * u_E + G_W * u_W + G_N * u_N + G_S * u_S
        denominator = G_E + G_W + G_N + G_S

        u_new = numerator / denominator

        # Update only inside Ω
        u[mask] = u_new[mask]
        u[~mask] = image[~mask]
        np.clip(u, 0.0, 1.0, out=u)

        # Convergence check
        max_change = float(np.max(np.abs(u[mask] - u_old[mask]))) if np.any(mask) else 0.0
        residuals.append(max_change)

        if verbose and (iteration + 1) % 500 == 0:
            print(f"  TV Iter {iteration + 1:5d} | max Δu = {max_change:.6f}")

        if max_change < tol:
            if verbose:
                print(f"  TV converged at iteration {iteration + 1}")
            break
    else:
        if verbose:
            print(f"  TV reached max_iter={max_iter}")

    return u, residuals

print("✓ TV inpainting defined")

### 1.3 Curvature-Driven Diffusion (CDD) Inpainting

Based on Chan & Shen (2001). Uses **G = g(|κ|)/|∇u|** where:
- κ is the isophote curvature
- g(s) = s^p with p ≥ 1

**Key insight:** g(0) = 0 ensures diffusion stops at straight edges!

In [ ]:
from scipy.ndimage import gaussian_filter, distance_transform_edt, binary_dilation, label
import numpy as np

def universal_cdd_inpaint(image, mask, mode='auto', verbose=True):
    """
    Universal CDD Inpainting - works for both synthetic shapes and photos.

    The mathematical model is the same:
        ∇·(G∇u) = 0  in Ω,  u = f on ∂Ω

    where G = g(|κ|)/|∇u| and g(s) = |s|^p

    Parameters are adapted based on image type:
    - Binary/synthetic: Mean init → Harmonic → Strong CDD (connects structures)
    - Photo/continuous: Nearest init → Sharp TV → Light CDD (preserves edges)

    Parameters
    ----------
    image : ndarray
        Input image (grayscale or RGB)
    mask : ndarray
        Boolean mask where True = inpaint region
    mode : str
        'auto' (detect), 'structure' (force connection mode), 'photo' (force edge preservation)
    """

    # Handle RGB vs grayscale
    is_rgb = len(image.shape) == 3

    if is_rgb:
        return _universal_cdd_rgb(image, mask, mode, verbose)
    else:
        return _universal_cdd_gray(image, mask, mode, verbose)


def _detect_image_type(image, mask):
    """
    Detect if image is binary/synthetic or continuous/photo.
    Returns 'structure' or 'photo'.
    """
    # Get values outside the mask
    if len(image.shape) == 3:
        outside_vals = image[~mask].flatten()
    else:
        outside_vals = image[~mask]

    # Count unique values (with some tolerance for near-binary)
    unique_vals = len(np.unique(np.round(outside_vals, 2)))

    # Binary images have very few unique values
    if unique_vals <= 10:
        return 'structure'
    else:
        return 'photo'


def _universal_cdd_gray(image, mask, mode='auto', verbose=True):
    """Universal CDD for grayscale images."""

    # Auto-detect mode if needed
    if mode == 'auto':
        mode = _detect_image_type(image, mask)
        if verbose:
            print(f"Auto-detected mode: {mode}")

    # Analyze mask geometry
    dist_map = distance_transform_edt(mask)
    max_gap_radius = np.max(dist_map) if np.any(mask) else 0

    if verbose:
        print(f"Gap radius: {max_gap_radius:.1f}px")

    u = image.copy().astype(np.float64)
    dilated = binary_dilation(mask)
    boundary = dilated & ~mask

    # ============================================================
    # MODE-DEPENDENT INITIALIZATION
    # ============================================================
    if mode == 'structure':
        # Mean init: creates smooth gradient for CDD to follow
        if np.any(boundary):
            u[mask] = image[boundary].mean()
        else:
            u[mask] = 0.5
        if verbose:
            print("  Init: Boundary mean (for structure connection)")
    else:
        # Nearest-boundary init: preserves edge structure
        if np.any(boundary):
            dist, indices = distance_transform_edt(~boundary, return_indices=True)
            my, mx = np.where(mask)
            u[my, mx] = image[indices[0][my, mx], indices[1][my, mx]]
        if verbose:
            print("  Init: Nearest-boundary (for edge preservation)")

    # ============================================================
    # MODE-DEPENDENT PARAMETERS
    # ============================================================
    if mode == 'structure':
        # Harmonic warm-up + Strong CDD
        if max_gap_radius <= 2.5:
            phase1_iters, sigma, p, cdd_iters = 50, 0.5, 1.0, 3000
        elif max_gap_radius <= 8.0:
            phase1_iters = int(100 * max_gap_radius)
            sigma, p, cdd_iters = 1.0, 1.5, 6000
        else:
            phase1_iters, sigma, p, cdd_iters = 1500, max_gap_radius/3.0, 2.0, 10000

        eps_phase1 = None  # Harmonic (equal weights)

    else:  # photo mode
        # Sharp TV + Light CDD
        if max_gap_radius <= 2.0:
            phase1_iters, sigma, p, cdd_iters = 100, 0.0, 1.0, 500
        elif max_gap_radius <= 5.0:
            phase1_iters, sigma, p, cdd_iters = 300, 0.2, 1.0, 1000
        else:
            phase1_iters, sigma, p, cdd_iters = 500, 0.5, 1.0, 2000

        eps_phase1 = 0.001  # Sharp TV

    # ============================================================
    # PHASE 1: Warm-up (Harmonic or Sharp TV)
    # ============================================================
    if phase1_iters > 0:
        if eps_phase1 is None:
            # HARMONIC: equal weights
            if verbose:
                print(f"  Phase 1: Harmonic ({phase1_iters} iters)")
            for _ in range(phase1_iters):
                u_pad = np.pad(u, 1, mode='edge')
                u_new = 0.25 * (u_pad[:-2,1:-1] + u_pad[2:,1:-1] +
                                u_pad[1:-1,:-2] + u_pad[1:-1,2:])
                u[mask] = u_new[mask]
        else:
            # SHARP TV: weighted by 1/|∇u|
            if verbose:
                print(f"  Phase 1: Sharp TV ({phase1_iters} iters, eps={eps_phase1})")
            eps_sq = eps_phase1 ** 2
            for _ in range(phase1_iters):
                u_pad = np.pad(u, 1, mode='edge')

                diff_E = u_pad[1:-1, 2:] - u_pad[1:-1, 1:-1]
                diff_W = u_pad[1:-1, :-2] - u_pad[1:-1, 1:-1]
                diff_N = u_pad[:-2, 1:-1] - u_pad[1:-1, 1:-1]
                diff_S = u_pad[2:, 1:-1] - u_pad[1:-1, 1:-1]

                G_E = 1.0 / np.sqrt(diff_E**2 + eps_sq)
                G_W = 1.0 / np.sqrt(diff_W**2 + eps_sq)
                G_N = 1.0 / np.sqrt(diff_N**2 + eps_sq)
                G_S = 1.0 / np.sqrt(diff_S**2 + eps_sq)

                u_new = (G_E*u_pad[1:-1,2:] + G_W*u_pad[1:-1,:-2] +
                         G_N*u_pad[:-2,1:-1] + G_S*u_pad[2:,1:-1]) / (G_E+G_W+G_N+G_S)
                u[mask] = u_new[mask]
                np.clip(u, 0, 1, out=u)

    # ============================================================
    # PHASE 2: CDD (curvature-driven diffusion)
    # Same mathematical model for both modes, different parameters
    # ============================================================
    if cdd_iters > 0:
        if verbose:
            print(f"  Phase 2: CDD ({cdd_iters} iters, p={p}, σ={sigma:.1f})")

        eps_cdd = 1e-4
        dt = 0.02

        for iteration in range(cdd_iters):
            u_old = u.copy()

            us = gaussian_filter(u, sigma=sigma) if sigma > 0 else u

            ux = (np.roll(us, -1, axis=1) - np.roll(us, 1, axis=1)) / 2.0
            uy = (np.roll(us, -1, axis=0) - np.roll(us, 1, axis=0)) / 2.0
            uxx = np.roll(us, -1, axis=1) - 2.0*us + np.roll(us, 1, axis=1)
            uyy = np.roll(us, -1, axis=0) - 2.0*us + np.roll(us, 1, axis=0)
            uxy = (np.roll(ux, -1, axis=0) - np.roll(ux, 1, axis=0)) / 2.0

            grad_mag = np.sqrt(ux**2 + uy**2 + eps_cdd**2)
            kappa = (uxx*uy**2 - 2*ux*uy*uxy + uyy*ux**2) / (grad_mag**3)

            # G = g(|κ|) / |∇u|, where g(s) = |s|^p
            G = (np.abs(kappa) ** p) / grad_mag
            G = np.clip(G, 0, 10.0)

            G_pad = np.pad(G, 1, mode='edge')
            G_E = 0.5 * (G_pad[1:-1, 2:] + G_pad[1:-1, 1:-1])
            G_W = 0.5 * (G_pad[1:-1, :-2] + G_pad[1:-1, 1:-1])
            G_N = 0.5 * (G_pad[:-2, 1:-1] + G_pad[1:-1, 1:-1])
            G_S = 0.5 * (G_pad[2:, 1:-1] + G_pad[1:-1, 1:-1])

            u_pad = np.pad(u, 1, mode='edge')
            divergence = (G_E * (u_pad[1:-1,2:] - u_pad[1:-1,1:-1]) +
                         G_W * (u_pad[1:-1,:-2] - u_pad[1:-1,1:-1]) +
                         G_N * (u_pad[:-2,1:-1] - u_pad[1:-1,1:-1]) +
                         G_S * (u_pad[2:,1:-1] - u_pad[1:-1,1:-1]))

            u[mask] = u[mask] + dt * divergence[mask]
            np.clip(u, 0, 1, out=u)

            if np.max(np.abs(u[mask] - u_old[mask])) < 1e-5:
                if verbose:
                    print(f"    Converged at iter {iteration + 1}")
                break

    # Binarize if synthetic
    if mode == 'structure' and len(np.unique(image[~mask])) <= 3:
        u = np.where(u > 0.5, 1.0, 0.0)

    return u, []


def _universal_cdd_rgb(image_rgb, mask, mode='auto', verbose=True):
    """Universal CDD for RGB images (processes each channel with coupled luminance)."""

    if mode == 'auto':
        mode = _detect_image_type(image_rgb, mask)
        if verbose:
            print(f"Auto-detected mode: {mode}")

    # For RGB, process per-channel but use shared luminance for curvature
    result = np.zeros_like(image_rgb, dtype=np.float64)

    # Simple approach: process as grayscale then apply per-channel
    # (For full RGB coupling, use the cdd_inpaint_rgb function)
    for c in range(3):
        if verbose and c == 0:
            result[:,:,c], _ = _universal_cdd_gray(image_rgb[:,:,c], mask, mode, verbose)
        else:
            result[:,:,c], _ = _universal_cdd_gray(image_rgb[:,:,c], mask, mode, verbose=False)

    return result, []


print("✓ Universal CDD inpainting defined")

### 1.4 Quick CDD (QCDD) Inpainting

Removes the |∇u| denominator: **G = g(|κ|)**

Faster convergence because g(|κ|)/|∇u| cancellation at corners is avoided.

In [ ]:
def qcdd_inpaint(
    image: np.ndarray,
    mask: np.ndarray,
    max_iter: int = 4000,
    eps: float = 1e-4,
    p: float = 1.0,
    sigma: float = 1.0,
    warmup_iters: int = 500,
    dt: float = 0.5,
    tol: float = 1e-5,
    verbose: bool = True,
) -> tuple:
    """
    QCDD (Quick CDD) - Curvature-driven diffusion WITHOUT the 1/|∇u| term.

    CDD:  G = g(|κ|) / |∇u|
    QCDD: G = g(|κ|)         <-- simpler, faster, more stable

    Same connectivity principle, just faster convergence.
    """
    u = image.copy().astype(np.float64)

    # Initialize with boundary mean (same as CDD)
    dilated = binary_dilation(mask)
    boundary = dilated & ~mask
    if np.any(boundary):
        u[mask] = image[boundary].mean()
    else:
        u[mask] = 0.5

    # Phase 1: Harmonic warm-up (same as CDD)
    if warmup_iters > 0:
        if verbose:
            print(f"  Phase 1: Harmonic warm-up ({warmup_iters} iters)")
        for _ in range(warmup_iters):
            u_pad = np.pad(u, 1, mode='edge')
            u_new = 0.25 * (u_pad[:-2,1:-1] + u_pad[2:,1:-1] +
                           u_pad[1:-1,:-2] + u_pad[1:-1,2:])
            u[mask] = u_new[mask]

    # Phase 2: QCDD
    if verbose:
        print(f"  Phase 2: QCDD ({max_iter} iters, p={p}, σ={sigma})")

    residuals = []

    for iteration in range(max_iter):
        u_old = u.copy()

        # Smooth for curvature computation
        us = gaussian_filter(u, sigma=sigma) if sigma > 0 else u

        # Compute derivatives
        ux = (np.roll(us, -1, axis=1) - np.roll(us, 1, axis=1)) / 2.0
        uy = (np.roll(us, -1, axis=0) - np.roll(us, 1, axis=0)) / 2.0
        uxx = np.roll(us, -1, axis=1) - 2.0*us + np.roll(us, 1, axis=1)
        uyy = np.roll(us, -1, axis=0) - 2.0*us + np.roll(us, 1, axis=0)
        uxy = (np.roll(ux, -1, axis=0) - np.roll(ux, 1, axis=0)) / 2.0

        # Curvature
        grad_mag = np.sqrt(ux**2 + uy**2 + eps**2)
        kappa = (uxx*uy**2 - 2*ux*uy*uxy + uyy*ux**2) / (grad_mag**3)

        # QCDD Conductivity: G = g(|κ|) = |κ|^p
        # NO baseline, NO division by gradient (that's what makes it "Quick")
        G = np.abs(kappa) ** p

        # Clip for stability (but no baseline!)
        G = np.clip(G, 0, 10.0)

        # Average to edges
        G_pad = np.pad(G, 1, mode='edge')
        G_E = 0.5 * (G_pad[1:-1, 2:] + G_pad[1:-1, 1:-1])
        G_W = 0.5 * (G_pad[1:-1, :-2] + G_pad[1:-1, 1:-1])
        G_N = 0.5 * (G_pad[:-2, 1:-1] + G_pad[1:-1, 1:-1])
        G_S = 0.5 * (G_pad[2:, 1:-1] + G_pad[1:-1, 1:-1])

        u_pad = np.pad(u, 1, mode='edge')

        # Semi-implicit update (stable with larger dt)
        sum_G_u = (G_E * u_pad[1:-1, 2:] +
                   G_W * u_pad[1:-1, :-2] +
                   G_N * u_pad[:-2, 1:-1] +
                   G_S * u_pad[2:, 1:-1])

        sum_G = G_E + G_W + G_N + G_S + 1e-10  # Small eps to avoid division by zero

        u_new = (u + dt * sum_G_u) / (1.0 + dt * sum_G)

        u[mask] = u_new[mask]
        u[~mask] = image[~mask]
        np.clip(u, 0, 1, out=u)

        max_change = float(np.max(np.abs(u[mask] - u_old[mask]))) if np.any(mask) else 0.0
        residuals.append(max_change)

        if verbose and (iteration + 1) % 500 == 0:
            print(f"    Iter {iteration + 1:5d} | max Δu = {max_change:.6f}")

        if max_change < tol:
            if verbose:
                print(f"    Converged at iter {iteration + 1}")
            break

    return u, residuals


def universal_qcdd_inpaint(image, mask, verbose=True):
    """
    Adaptive QCDD wrapper (same logic as universal_cdd_inpaint).
    """
    dist_map = distance_transform_edt(mask)
    max_gap_radius = np.max(dist_map) if np.any(mask) else 0

    if verbose:
        print(f"Gap radius: {max_gap_radius:.1f}px")

    # Same adaptive parameters as CDD
    if max_gap_radius <= 2.5:
        if verbose: print("Regime: Micro")
        warmup, sigma, p, iters = 50, 0.5, 1.0, 2000
    elif max_gap_radius <= 8.0:
        if verbose: print("Regime: Medium")
        warmup = int(100 * max_gap_radius)
        sigma, p, iters = 1.0, 1.5, 4000
    else:
        if verbose: print("Regime: Large")
        warmup = 1500
        sigma, p, iters = max_gap_radius/3.0, 2.0, 6000

    res, residuals = qcdd_inpaint(
        image, mask,
        warmup_iters=warmup,
        max_iter=iters,
        sigma=sigma,
        p=p,
        dt=0.5,  # Larger dt since QCDD is more stable
        verbose=verbose
    )

    # Binarize if synthetic
    if len(np.unique(image[~mask])) <= 3:
        res = np.where(res > 0.5, 1.0, 0.0)

    return res, residuals


print("✓ QCDD inpainting defined")

# Helper Functions

In [ ]:
# CELL 2: Helper functions
from PIL import Image

def load_image(path, grayscale=False):
    """Load an image from disk as a float32 array in [0, 1]."""
    mode = 'L' if grayscale else 'RGB'
    img  = Image.open(path).convert(mode)
    return np.array(img, dtype=np.float32) / 255.0


def show_images(images, titles, cmap=None, figsize=None):
    """Quick multi-panel display helper."""
    n = len(images)
    if figsize is None:
        figsize = (4 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap or 'gray', vmin=0, vmax=1)
        else:
            ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('Helpers defined')

In [ ]:
def run_all_methods(damaged, mask, verbose=False):
    """Run all four inpainting methods and return results."""
    results = {}

    print("Running Harmonic...")
    results['Harmonic'], _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=verbose)

    print("Running TV...")
    results['TV'], _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=verbose)

    print("Running CDD...")
    results['CDD'], _ = cdd_inpaint_two_phase(damaged, mask, max_iter=2000,
                                           harmonic_init_iters=200, verbose=verbose)

    print("Running QCDD...")
    results['QCDD'], _ = qcdd_inpaint(damaged, mask, max_iter=2000,
                                       harmonic_init_iters=200, verbose=verbose)

    print("Done!\n")
    return results


def visualize_comparison(original, damaged, mask, results, title="Comparison"):
    """Create a comparison figure showing all methods."""
    fig = plt.figure(figsize=(14, 8))
    gs = GridSpec(2, 4, figure=fig, hspace=0.3, wspace=0.1)

    # Top row: Original, Damaged, Mask
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(original, cmap='gray', vmin=0, vmax=1)
    ax1.set_title('Original', fontsize=12)
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(damaged, cmap='gray', vmin=0, vmax=1)
    ax2.set_title('Damaged', fontsize=12)
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[0, 2])
    overlay = np.stack([damaged]*3, axis=-1)
    overlay[mask] = [1, 0, 0]  # Red for masked region
    ax3.imshow(overlay)
    ax3.set_title('Mask (red)', fontsize=12)
    ax3.axis('off')

    ax4 = fig.add_subplot(gs[0, 3])
    ax4.axis('off')
    ax4.text(0.5, 0.5, f'{title}\n\nModels tested:\n• Harmonic\n• TV\n• CDD\n• QCDD',
             ha='center', va='center', fontsize=11, transform=ax4.transAxes)

    # Bottom row: Results
    method_names = ['Harmonic', 'TV', 'CDD', 'QCDD']
    for idx, name in enumerate(method_names):
        ax = fig.add_subplot(gs[1, idx])
        ax.imshow(results[name], cmap='gray', vmin=0, vmax=1)
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    return fig

print("✓ Helper functions defined")

In [ ]:
def create_broken_bar(size=64, bar_width=8, gap_width=20):
    """Create a horizontal bar with a gap in the middle."""
    img = np.zeros((size, size))  # Black background

    bar_top = size // 2 - bar_width // 2
    bar_bottom = bar_top + bar_width
    img[bar_top:bar_bottom, :] = 1.0  # White bar

    mask = np.zeros((size, size), dtype=bool)
    gap_left = size // 2 - gap_width // 2
    gap_right = gap_left + gap_width
    mask[bar_top-2:bar_bottom+2, gap_left:gap_right] = True

    damaged = img.copy()
    damaged[mask] = 0.5 + 0.1 * np.random.randn(np.sum(mask))

    return img, damaged, mask


def create_broken_ring(size=64, radius=20, thickness=4, gap_angle=60):
    """Create a ring with a gap. Classic CDD test from Chan & Shen."""
    img = np.zeros((size, size))  # Black background
    y, x = np.ogrid[:size, :size]
    center = size // 2

    dist = np.sqrt((x - center)**2 + (y - center)**2)
    ring = (dist >= radius - thickness/2) & (dist <= radius + thickness/2)
    img[ring] = 1.0  # White ring

    angle = np.arctan2(y - center, x - center) * 180 / np.pi
    gap_half = gap_angle / 2
    in_gap = (angle > 90 - gap_half) & (angle < 90 + gap_half)

    mask = ring & in_gap

    damaged = img.copy()
    damaged[mask] = 0.5 + 0.1 * np.random.randn(np.sum(mask))

    return img, damaged, mask


def create_crossing_lines(size=64, line_width=4, gap_size=12):
    """Create two crossing diagonal lines with a gap at intersection."""
    img = np.zeros((size, size))  # Black background

    for i in range(size):
        for j in range(size):
            if abs(i - j) < line_width:
                img[i, j] = 1.0  # White lines
            if abs(i - (size - 1 - j)) < line_width:
                img[i, j] = 1.0

    mask = np.zeros((size, size), dtype=bool)
    center = size // 2
    half_gap = gap_size // 2
    mask[center-half_gap:center+half_gap, center-half_gap:center+half_gap] = True

    damaged = img.copy()
    damaged[mask] = 0.5 + 0.1 * np.random.randn(np.sum(mask))

    return img, damaged, mask

def create_ring_with_cross(size=64, radius=20, thickness=5, gap_angle=70):
    """
    Classic CDD test: ring with a + sign as corruption in the gap.
    TV will follow the cross and produce 4 arcs.
    CDD should ignore the cross and complete the ring.
    """
    img = np.zeros((size, size))
    y, x = np.ogrid[:size, :size]
    center = size // 2

    # Create ring
    dist = np.sqrt((x - center)**2 + (y - center)**2)
    ring = (dist >= radius - thickness/2) & (dist <= radius + thickness/2)
    img[ring] = 1.0

    # Create mask (gap at bottom)
    angle = np.arctan2(y - center, x - center) * 180 / np.pi
    gap_half = gap_angle / 2
    in_gap = (angle > -90 - gap_half) & (angle < -90 + gap_half)  # Bottom of ring
    mask = ring & in_gap

    # Create damaged image with + sign as corruption inside the gap
    damaged = img.copy()

    # Draw + sign inside masked region
    cross_half = int(thickness * 1.2)
    gap_center_y = center + radius  # Bottom of ring

    # Vertical bar of +
    damaged[gap_center_y - cross_half : gap_center_y + cross_half,
            center - 1 : center + 2] = 1.0
    # Horizontal bar of +
    damaged[gap_center_y - 1 : gap_center_y + 2,
            center - cross_half : center + cross_half] = 1.0

    return img, damaged, mask

print("✓ Test image generators defined")

# Testing Simple Shapes

In [ ]:
# ==========================================
# CELL 4: The Ring with a Cross (Topology Test)
# ==========================================
import numpy as np
import matplotlib.pyplot as plt

def create_ring_with_cross(size=100, radius=30, thickness=8, cross_thickness=14):
    # 1. White background
    img = np.ones((size, size))

    Y, X = np.ogrid[:size, :size]
    dist_from_center = np.sqrt((X - size//2)**2 + (Y - size//2)**2)

    # 2. Black ring
    ring_mask = (dist_from_center >= radius - thickness/2) & (dist_from_center <= radius + thickness/2)
    img[ring_mask] = 0.0

    # 3. The Cross Mask
    mask = np.zeros((size, size), dtype=bool)
    # Vertical bar
    mask[:, size//2 - cross_thickness//2 : size//2 + cross_thickness//2] = True
    # Horizontal bar
    mask[size//2 - cross_thickness//2 : size//2 + cross_thickness//2, :] = True

    damaged = img.copy()
    damaged[mask] = 0.5 # Fill the missing cross with gray
    return img, damaged, mask

original, damaged, mask = create_ring_with_cross()

print("Running Harmonic...")
h_res, _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running TV...")
tv_res, _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running CDD...")
# The universal wrapper calculates the gap radius (~7-10px)
# and automatically adjusts sigma and iterations to rebuild the ring structure.
cdd_res, _ = universal_cdd_inpaint(damaged, mask, verbose=True)

print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

# Plotting
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic\n(Blurs)", "TV\n(Straightens/Pinches)", "CDD\n(Rebuilds Ring)", "QCDD"]
images = [damaged, h_res, tv_res, cdd_res, qcdd_res]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')

plt.suptitle("Test 4: The Ring with a Cross (Complex Topology)", fontsize=16)
plt.show()

In [ ]:
# ==========================================
# CELL 2: The Broken Ring (Curve Following)
# ==========================================
import numpy as np
import matplotlib.pyplot as plt

def create_broken_ring(size=80, radius=25, thickness=6):
    Y, X = np.ogrid[:size, :size]
    dist_from_center = np.sqrt((X - size//2)**2 + (Y - size//2)**2)

    img = np.zeros((size, size))
    ring_mask = (dist_from_center >= radius - thickness/2) & (dist_from_center <= radius + thickness/2)
    img[ring_mask] = 1.0

    mask = np.zeros((size, size), dtype=bool)
    # Target a specific wedge of the ring
    mask[size//2 + 15 : size//2 + 30, size//2 - 10 : size//2 + 10] = True

    damaged = img.copy()
    damaged[mask] = 0.5
    return img, damaged, mask

original, damaged, mask = create_broken_ring()

print("Running Harmonic...")
h_res, _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running TV...")
tv_res, _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running CDD...")
# The universal wrapper detects the gap size and applies:
# 1. Long harmonic bridge to find the curve trajectory.
# 2. High 'p' value to fight the "Chord Effect" on the inner ring.
# 3. Automatic thresholding for the final result.
cdd_res, _ = universal_cdd_inpaint(damaged, mask, verbose=True)

print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

# Plotting
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic", "TV\n(Flattens/Pinches)", "CDD\n(Rebuilds Curve)", "QCDD"]
images = [damaged, h_res, tv_res, cdd_res, qcdd_res]
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Test 2: Curvature Trajectory (The Broken Ring)", fontsize=16)
plt.show()

In [ ]:
# ==========================================
# CELL 1: The Broken Bar (Connectivity Test)
# ==========================================
import numpy as np
import matplotlib.pyplot as plt

def create_broken_bar(size=64, bar_width=8, gap_width=20):
    img = np.zeros((size, size))
    img[size//2 - bar_width//2 : size//2 + bar_width//2, :] = 1.0

    mask = np.zeros((size, size), dtype=bool)
    mask[:, size//2 - gap_width//2 : size//2 + gap_width//2] = True

    damaged = img.copy()
    damaged[mask] = 0.5
    return img, damaged, mask

original, damaged, mask = create_broken_bar(size=64, bar_width=10, gap_width=12)

print("Running Harmonic...")
h_res, _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running TV...")
tv_res, _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running CDD...")
# The universal wrapper autonomously calculates parameters based on gap width
# and automatically applies the academic binarization threshold.
cdd_res, _ = universal_cdd_inpaint(damaged, mask, verbose=True)

print("Running QCDD...")
qcdd_res, _ = universal_qcdd_inpaint(damaged, mask, verbose=True)

# Plotting
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic\n(Blurs)", "TV\n(Pinches off)", "CDD\n(Connects)", "QCDD"]
images = [damaged, h_res, tv_res, cdd_res, qcdd_res]
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Test 1: The Connectivity Principle (Straight Line)", fontsize=16)
plt.show()

In [ ]:
# ==========================================
# CELL 3: Crossing Lines (Intersection Test)
# ==========================================
import numpy as np
import matplotlib.pyplot as plt

def create_crossing_lines(size=64, line_width=6, gap_size=16):
    # 1. Black background
    img = np.zeros((size, size))

    # 2. Draw two crossing white lines (Main diagonal and Anti-diagonal)
    for i in range(size):
        for j in range(size):
            # Main diagonal: |i - j| < width
            if abs(i - j) < line_width:
                img[i, j] = 1.0
            # Anti-diagonal: |i + j - size| < width
            if abs(i + j - size) < line_width:
                img[i, j] = 1.0

    # 3. Create a square mask in the center where they intersect
    mask = np.zeros((size, size), dtype=bool)
    center = size // 2
    half_gap = gap_size // 2
    mask[center - half_gap : center + half_gap,
         center - half_gap : center + half_gap] = True

    damaged = img.copy()
    damaged[mask] = 0.5 # Initial gray fill for the missing intersection
    return img, damaged, mask

original, damaged, mask = create_crossing_lines(size=80, line_width=6, gap_size=18)

print("Running Harmonic...")
h_res, _ = harmonic_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running TV...")
tv_res, _ = tv_inpaint(damaged, mask, max_iter=2000, verbose=False)

print("Running CDD...")
# The universal wrapper analyzes the center gap and applies the correct
# curvature-driven logic to bridge the intersection.
cdd_res, _ = universal_cdd_inpaint(damaged, mask, verbose=True)

print("Running QCDD...")
qcdd_res, _ = qcdd_inpaint(
    damaged, mask, max_iter=4000,
    harmonic_init_iters=800, sigma_curvature=2.0, eps=0.1, p=1.5, verbose=False
)

# Plotting
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Damaged", "Harmonic\n(Blurs)", "TV\n(Blobs Center)", "CDD\n(Reconstructs X)", "QCDD"]
images = [damaged, h_res, tv_res, cdd_res, qcdd_res]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')

plt.suptitle("Test 3: Crossing Lines (Intersection Topology)", fontsize=16)
plt.show()

# Testing on Photos

In [ ]:
# CELL 0: Colab Stuff

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/pde_v2')

In [ ]:
# CELL 8a: Interactive mask painter (HTML canvas — works in Colab)

import numpy as np
import base64, json
from PIL import Image as PILImage
from IPython.display import display, HTML
from google.colab import output
import io

IMAGE_PATH = '/content/drive/MyDrive/pde_v2/content/test_image_8.PNG'
original_rgb = load_image(IMAGE_PATH, grayscale=False)

# FIX: If the downscaled image is 2D, duplicate it into 3 channels for the RGB solver
if len(original_rgb.shape) == 2:
    original_rgb = np.stack((original_rgb, original_rgb, original_rgb), axis=-1)

H3, W3, _ = original_rgb.shape

# Encode image to base64 so we can display it in the canvas
img_uint8 = (np.clip(original_rgb, 0, 1) * 255).astype(np.uint8)
pil_img   = PILImage.fromarray(img_uint8)
buf       = io.BytesIO()
pil_img.save(buf, format='PNG')
img_b64   = base64.b64encode(buf.getvalue()).decode('utf-8')

# We'll store the mask here after the user clicks "Save Mask"
painted_mask = np.zeros((H3, W3), dtype=bool)

def save_mask(mask_b64):
    global painted_mask
    mask_bytes = base64.b64decode(mask_b64)
    mask_img   = PILImage.open(io.BytesIO(mask_bytes)).convert('L')
    mask_arr   = np.array(mask_img)
    painted_mask = mask_arr > 128
    print(f'Mask saved: {painted_mask.sum()} pixels ({100*painted_mask.mean():.2f}%)')

output.register_callback('save_mask', save_mask)

DISPLAY_W = min(W3, 700)
DISPLAY_H = int(H3 * DISPLAY_W / W3)

html = f"""
<div style="font-family: monospace; user-select: none;">
  <div style="margin-bottom:8px; display:flex; gap:12px; align-items:center;">
    <label>Brush size:
      <input id="brushRange" type="range" min="2" max="60" value="12"
             oninput="document.getElementById('brushVal').textContent=this.value">
      <span id="brushVal">12</span>px
    </label>
    <button onclick="clearMask()"
      style="padding:4px 12px; cursor:pointer;">Clear</button>
    <button onclick="saveMask()"
      style="padding:4px 14px; background:#2a7; color:#fff;
             border:none; border-radius:4px; cursor:pointer; font-size:14px;">
      Save Mask ✓
    </button>
    <span id="status" style="color:#888;"></span>
  </div>

  <div style="position:relative; width:{DISPLAY_W}px; height:{DISPLAY_H}px;">
    <!-- background: the original image -->
    <canvas id="imgCanvas" width="{DISPLAY_W}" height="{DISPLAY_H}"
            style="position:absolute; top:0; left:0;"></canvas>
    <!-- foreground: the red painted mask -->
    <canvas id="drawCanvas" width="{DISPLAY_W}" height="{DISPLAY_H}"
            style="position:absolute; top:0; left:0; opacity:0.45; cursor:crosshair;"></canvas>
  </div>
</div>

<script>
const IMG_W = {W3}, IMG_H = {H3};
const DISP_W = {DISPLAY_W}, DISP_H = {DISPLAY_H};

// Draw the photo onto the background canvas
const imgCanvas  = document.getElementById('imgCanvas');
const imgCtx     = imgCanvas.getContext('2d');
const drawCanvas = document.getElementById('drawCanvas');
const drawCtx    = drawCanvas.getContext('2d');

const photo = new Image();
photo.onload = () => imgCtx.drawImage(photo, 0, 0, DISP_W, DISP_H);
photo.src = 'data:image/png;base64,{img_b64}';

drawCtx.fillStyle = 'red';

let painting = false;

function getBrush() {{
  return parseInt(document.getElementById('brushRange').value);
}}

function getPos(e) {{
  const rect = drawCanvas.getBoundingClientRect();
  const clientX = e.touches ? e.touches[0].clientX : e.clientX;
  const clientY = e.touches ? e.touches[0].clientY : e.clientY;
  return {{
    x: clientX - rect.left,
    y: clientY - rect.top
  }};
}}

function paint(e) {{
  if (!painting) return;
  e.preventDefault();
  const {{x, y}} = getPos(e);
  const r = getBrush();
  drawCtx.beginPath();
  drawCtx.arc(x, y, r, 0, Math.PI * 2);
  drawCtx.fill();
}}

drawCanvas.addEventListener('mousedown',  e => {{ painting = true;  paint(e); }});
drawCanvas.addEventListener('mousemove',  paint);
drawCanvas.addEventListener('mouseup',    () => painting = false);
drawCanvas.addEventListener('mouseleave', () => painting = false);

function clearMask() {{
  drawCtx.clearRect(0, 0, DISP_W, DISP_H);
  document.getElementById('status').textContent = 'Cleared.';
}}

function saveMask() {{
  // Render mask at FULL resolution (not just display size)
  const fullCanvas = document.createElement('canvas');
  fullCanvas.width  = IMG_W;
  fullCanvas.height = IMG_H;
  const fullCtx = fullCanvas.getContext('2d');

  // Scale the drawn mask up to native image resolution
  fullCtx.drawImage(drawCanvas, 0, 0, IMG_W, IMG_H);

  // Export as grayscale: convert red pixels → white, rest → black
  const imageData = fullCtx.getImageData(0, 0, IMG_W, IMG_H);
  const grayCanvas = document.createElement('canvas');
  grayCanvas.width  = IMG_W;
  grayCanvas.height = IMG_H;
  const grayCtx = grayCanvas.getContext('2d');
  const grayData = grayCtx.createImageData(IMG_W, IMG_H);

  for (let i = 0; i < imageData.data.length; i += 4) {{
    const alpha = imageData.data[i + 3];   // painted pixels have alpha > 0
    const val   = alpha > 10 ? 255 : 0;
    grayData.data[i]     = val;
    grayData.data[i + 1] = val;
    grayData.data[i + 2] = val;
    grayData.data[i + 3] = 255;
  }}
  grayCtx.putImageData(grayData, 0, 0);

  const maskB64 = grayCanvas.toDataURL('image/png').split(',')[1];
  document.getElementById('status').textContent = 'Mask sent to Python...';
  google.colab.kernel.invokeFunction('save_mask', [maskB64], {{}});
  document.getElementById('status').textContent = 'Saved ✓';
}}
</script>
"""

display(HTML(html))
print(f'Image: {H3}x{W3}  — Paint over damage, then click "Save Mask ✓", then run Cell 8b.')

In [ ]:
# Cell: Run Adaptive CDD on Photo
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# --- EXECUTION ---
if 'painted_mask' not in globals() or painted_mask.sum() == 0:
    print('Mask is empty — paint in Cell 8a and click Save Mask first.')
else:
    print(f"Running Adaptive CDD Inpainting on {IMAGE_PATH}...")

    # Use universal function with photo mode
    restored_rgb, _ = universal_cdd_inpaint(original_rgb, painted_mask, mode='photo', verbose=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original_rgb)
    axes[0].set_title("Original Photo")

    mask_overlay = original_rgb.copy()
    mask_overlay[painted_mask] = [1, 0, 0]
    axes[1].imshow(mask_overlay)
    axes[1].set_title("Your Painted Mask")

    axes[2].imshow(np.clip(restored_rgb, 0, 1))
    axes[2].set_title("Adaptive CDD (Sharp TV Init)")

    for ax in axes: ax.axis('off')
    plt.show()

    out = (np.clip(restored_rgb, 0, 1) * 255).astype('uint8')
    out_path = '/content/restored_cdd.jpg'
    PILImage.fromarray(out, mode='RGB').save(out_path)
    print(f'Saved to: {out_path}')